# DATA209 — Advanced Exploratory Data Analysis
# Practical P7-8 · Univariate EDA

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 4 · Module 2 · CO2

---

**Objective.** Run a complete univariate sweep, compare the available distribution plots on the same variable, and interpret the findings.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.


### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

Recap from P1-2 — the dataset, the target and the numeric column list.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P7-8 — Univariate EDA

### Perform univariate EDA

A univariate sweep answers four questions for every column: **what is the centre, what is the
spread, what is the shape, and what is unusual?** Write it once as a function and reuse it in
every project including the final one.

In [ ]:
# ---- A reusable univariate sweep ---------------------------------------
def univariate_sweep(frame, cols=None):
    """Centre, spread, shape and anomaly flags for every numeric column."""
    cols = cols or frame.select_dtypes(include=[np.number]).columns.tolist()
    out = []
    for c in cols:
        s   = frame[c].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        out.append({
            "column"    : c,
            "n"         : len(s),
            "mean"      : s.mean(),
            "median"    : s.median(),
            "std"       : s.std(),
            "IQR"       : iqr,
            "skew"      : s.skew(),
            "kurtosis"  : s.kurtosis(),
            "zeros_%"   : (s == 0).mean() * 100,
            "iqr_out_%" : (((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).mean() * 100),
        })
    return pd.DataFrame(out).set_index("column")

sweep = univariate_sweep(df, numeric_cols)
print(sweep.round(2).to_string())

In [ ]:
# ---- Flag what needs attention ------------------------------------------
flags = pd.DataFrame({
    "strong_skew"   : sweep["skew"].abs() > 1,
    "heavy_tails"   : sweep["kurtosis"] > 3,
    "zero_inflated" : sweep["zeros_%"] > 50,
    "many_outliers" : sweep["iqr_out_%"] > 5,
})
flags["issues"] = flags.sum(axis=1)
print(flags.sort_values("issues", ascending=False).to_string())

print("\nColumns needing treatment before modelling:")
print(" ", flags[flags["issues"] >= 2].index.tolist())

### Compare different distribution plots

Five ways to show one variable. Each is honest; each hides something.

In [ ]:
# ---- Five views of one variable ----------------------------------------
col = "BounceRates"
s   = df[col]

fig, axes = plt.subplots(1, 5, figsize=(15, 2.9))

sns.histplot(s, bins=40, ax=axes[0], color="#1F6F6B")
axes[0].set_title("Histogram (40 bins)")

sns.histplot(s, bins=8, ax=axes[1], color="#1F6F6B")
axes[1].set_title("Histogram (8 bins)")

sns.kdeplot(s, ax=axes[2], fill=True, color="#1F6F6B")
axes[2].set_title("Density (KDE)")

sns.boxplot(x=s, ax=axes[3], color="#1F6F6B")
axes[3].set_title("Boxplot")

sns.ecdfplot(s, ax=axes[4], color="#1F6F6B")
axes[4].set_title("ECDF")

for a in axes: a.set_xlabel(col); a.set_ylabel("")
plt.tight_layout(); plt.show()

print("Same variable, five renderings:")
print("- 40 bins shows the spike at zero; 8 bins conceals it entirely.")
print("- The KDE invents density below zero, which is impossible for a rate.")
print("- The boxplot compresses everything into five numbers and a cloud of flagged points.")
print("- The ECDF needs no binning choice and reads percentiles exactly.")

In [ ]:
# ---- Univariate comparison across the target groups --------------------
show = ["PageValues", "ProductRelated", "BounceRates", "ExitRates"]
fig, axes = plt.subplots(2, len(show), figsize=(14, 5.6))

for j, c in enumerate(show):
    sns.kdeplot(data=df, x=c, hue="Revenue", ax=axes[0, j],
                fill=True, alpha=0.35, common_norm=False)
    axes[0, j].set_title(c); axes[0, j].set_ylabel("")
    sns.boxplot(data=df, x="Revenue", y=c, ax=axes[1, j], showfliers=False)
    axes[1, j].set_title("")

plt.tight_layout(); plt.show()

comparison = df.groupby("Revenue")[show].median().T
comparison.columns = ["no purchase", "purchase"]
comparison["ratio"] = comparison["purchase"] / comparison["no purchase"].replace(0, np.nan)
print(comparison.round(3).to_string())

### Interpret findings

A figure without a written interpretation earns no marks. Convert each observation into a claim
about the business question.

In [ ]:
# ---- Written interpretation --------------------------------------------
findings = [
    ("PageValues",
     "Median is 0 for the large majority of sessions but clearly positive for converting "
     "sessions. This is the strongest single separator in the dataset."),
    ("ProductRelated / _Duration",
     "Both are strongly right-skewed and zero-inflated. Converting sessions view more product "
     "pages and stay longer, but the tail is extreme — transform before modelling (P23-24)."),
    ("BounceRates / ExitRates",
     "Both spike at zero and are bounded in [0,1]. Converting sessions have visibly lower "
     "values. The two are near-duplicates of each other — check in P9-10."),
    ("SpecialDay",
     "Almost entirely zero. Nearly constant, so it carries little information on its own."),
    ("Administrative / Informational",
     "Low counts, heavily zero-inflated. Candidates for combining into a single "
     "'non-product page depth' feature in P25-26."),
]
for name, text in findings:
    print(f"- {name}\n    {textwrap.fill(text, 92, subsequent_indent='    ')}")

### Deliverable — P7-8

A notebook plus a **one-page findings table, one row per variable**, giving centre, spread, shape,
anomalies and a one-line interpretation.